# Quantitative Momentum Strategy

"Momentum investing" means investing in the stocks that have increased in price the most.

For this project, we're going to build an investing strategy that selects the 50 stocks with the highest price momentum. From there, we will calculate recommended trades for an equal-weight portfolio of these 50 stocks.


## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [1]:
import numpy as np
import pandas as pd
import requests
import math
from scipy import stats
import xlsxwriter
import yfinance as yf

## Importing Our List of Stocks

As before, we'll need to import our list of stocks and our API token before proceeding. Make sure the `.csv` file is still in your working directory and import it with the following command:

In [2]:
stocks = pd.read_csv('sp_500_stocks.csv')
stocks

,Ticker
0,A
1,AAL
2,AAP
3,AAPL
4,ABBV
...,...
500,YUM
501,ZBH
502,ZBRA
503,ZION


## Making Our First API Call

It's now time to make the first version of our momentum screener!

We need to get one-year price returns for each stock in the universe. Here's how.

In [10]:
tickers = stocks["Ticker"].dropna().tolist()
tickers = [ticker.strip() for ticker in tickers]


price_data = yf.download(
    tickers,
    period="2y",
    interval="1d",
    auto_adjust=True,
    group_by="ticker",
    threads=True,
    progress=False
)

price_data

data = []

for ticker in tickers:
    try:
        prices = price_data[ticker]["Close"].dropna()
        one_year_return = (prices.iloc[-1] / prices.iloc[0]) - 1

        data.append ({
            "Ticker": ticker,
            "Start Price": prices.iloc[0],
            "End Price": prices.iloc[-1],
            "One-Year Return": one_year_return,
            "One-Year Return %": one_year_return * 100
        })
    
    except Exception as e:
        data.append({
            "Ticker": ticker,
            "Start Price": None,
            "End Price": None,
            "One-Year Return": None,
            "One-Year Return %": None
        })

returns_df  = pd.DataFrame(data)
# readable number
# Optional sortieren (beste Performance zuerst)
returns_df = returns_df.sort_values("One-Year Return %", ascending=False)

returns_df

$ALXN: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$TIF: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$COG: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$MMC: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$BRK.B: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$CTL: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$CERN: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$MRO: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$KSU: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data fo

,Ticker,Start Price,End Price,One-Year Return,One-Year Return %
422,STX,86.180679,731.934998,7.493029,749.302887
479,WDC,54.639435,441.989990,7.089212,708.921234
202,GLW,32.097492,160.009995,3.985124,398.512449
324,MU,119.293228,578.340027,3.848054,384.805413
441,TPR,37.902439,140.589996,2.709260,270.925987
...,...,...,...,...,...
467,VIAC,NaN,NaN,NaN,NaN
478,WBA,NaN,NaN,NaN,NaN
484,WLTW,NaN,NaN,NaN,NaN
489,WRK,NaN,NaN,NaN,NaN


## Removing Low-Momentum Stocks

The investment strategy that we're building seeks to identify the 50 highest-momentum stocks in the S&P 500.

Because of this, the next thing we need to do is remove all the stocks in our DataFrame that fall below this momentum threshold. We'll sort the DataFrame by the stocks' one-year price return, and drop all stocks outside the top 50.


In [11]:
returns_df = returns_df[:50]
returns_df

,Ticker,Start Price,End Price,One-Year Return,One-Year Return %
422,STX,86.180679,731.934998,7.493029,749.302887
479,WDC,54.639435,441.989990,7.089212,708.921234
202,GLW,32.097492,160.009995,3.985124,398.512449
324,MU,119.293228,578.340027,3.848054,384.805413
441,TPR,37.902439,140.589996,2.709260,270.925987
50,AVGO,128.408066,411.769989,2.206730,220.672994
242,INTC,30.774776,96.669998,2.141209,214.120879
233,HWM,79.745659,239.625000,2.004866,200.486576
194,FTI,26.038755,74.470001,1.859968,185.996777
275,LB,22.931553,65.099998,1.838883,183.888312


## Calculating the Number of Shares to Buy

Just like in the last project, we now need to calculate the number of shares we need to buy. The one change we're going to make is wrapping this functionality inside a function, since we'll be using it again later in this Jupyter Notebook.

Since we've already done most of the work on this, try to complete the following two code cells without watching me do it first!

In [12]:
returns_df.reset_index(inplace = True)
returns_df

,index,Ticker,Start Price,End Price,One-Year Return,One-Year Return %
0,422,STX,86.180679,731.934998,7.493029,749.302887
1,479,WDC,54.639435,441.989990,7.089212,708.921234
2,202,GLW,32.097492,160.009995,3.985124,398.512449
3,324,MU,119.293228,578.340027,3.848054,384.805413
4,441,TPR,37.902439,140.589996,2.709260,270.925987
5,50,AVGO,128.408066,411.769989,2.206730,220.672994
6,242,INTC,30.774776,96.669998,2.141209,214.120879
7,233,HWM,79.745659,239.625000,2.004866,200.486576
8,194,FTI,26.038755,74.470001,1.859968,185.996777
9,275,LB,22.931553,65.099998,1.838883,183.888312


In [13]:
# investment
portfolio_size = 1000000
# Price as numeric
returns_df["End Price"] = pd.to_numeric(returns_df["End Price"], errors="coerce")
# clean invalid values
returns_df = returns_df.replace([np.inf, -np.inf], np.nan)

# how many of each stock of the index is bought based on the portfolio size (investment)
position_size = portfolio_size / len(returns_df.index)
print(position_size)
returns_df["Number of Shares to Buy"] = np.where(
    returns_df["End Price"] > 0,
    np.floor(position_size / returns_df["End Price"]),
    np.nan
)

returns_df

20000.0


C:\Users\nadir_i69tsxn\AppData\Local\Temp\ipykernel_1600\2044626268.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  returns_df["End Price"] = pd.to_numeric(returns_df["End Price"], errors="coerce")


,index,Ticker,Start Price,End Price,One-Year Return,One-Year Return %,Number of Shares to Buy
0,422,STX,86.180679,731.934998,7.493029,749.302887,27.0
1,479,WDC,54.639435,441.989990,7.089212,708.921234,45.0
2,202,GLW,32.097492,160.009995,3.985124,398.512449,124.0
3,324,MU,119.293228,578.340027,3.848054,384.805413,34.0
4,441,TPR,37.902439,140.589996,2.709260,270.925987,142.0
5,50,AVGO,128.408066,411.769989,2.206730,220.672994,48.0
6,242,INTC,30.774776,96.669998,2.141209,214.120879,206.0
7,233,HWM,79.745659,239.625000,2.004866,200.486576,83.0
8,194,FTI,26.038755,74.470001,1.859968,185.996777,268.0
9,275,LB,22.931553,65.099998,1.838883,183.888312,307.0


## Building a Better (and More Realistic) Momentum Strategy

Real-world quantitative investment firms differentiate between "high quality" and "low quality" momentum stocks:

* High-quality momentum stocks show "slow and steady" outperformance over long periods of time
* Low-quality momentum stocks might not show any momentum for a long time, and then surge upwards.

The reason why high-quality momentum stocks are preferred is because low-quality momentum can often be cause by short-term news that is unlikely to be repeated in the future (such as an FDA approval for a biotechnology company).

To identify high-quality momentum, we're going to build a strategy that selects stocks from the highest percentiles of: 

* 1-month price returns
* 3-month price returns
* 6-month price returns
* 1-year price returns

Let's start by building our DataFrame. You'll notice that I use the abbreviation `hqm` often. It stands for `high-quality momentum`.

In [15]:
from scipy.stats import percentileofscore

portfolio_size = 1000000


def calculate_return(prices, months):
    prices = prices.dropna()

    if prices.empty:
        return np.nan

    end_date = prices.index[-1]
    start_date = end_date - pd.DateOffset(months=months)

    past_prices = prices[prices.index <= start_date]

    if past_prices.empty:
        return np.nan

    start_price = past_prices.iloc[-1]
    end_price = prices.iloc[-1]

    if start_price == 0:
        return np.nan

    return (end_price / start_price) - 1

rows = []

for ticker in tickers:
    try:
        prices = price_data[ticker]["Close"].dropna()

        rows.append({
            "Ticker": ticker,
            "Price": float(prices.iloc[-1]),
            "Number of Shares To Buy": 0,
            "One-Year Price Return": float(calculate_return(prices, 12)),
            "One-Year Return Percentile": np.nan,
            "Six-Month Price Return": float(calculate_return(prices, 6)),
            "Six-Month Return Percentile": np.nan,
            "Three-Month Price Return": float(calculate_return(prices, 3)),
            "Three-Month Return Percentile": np.nan,
            "One-Month Price Return": float(calculate_return(prices, 1)),
            "One-Month Return Percentile": np.nan
        })
    
    except Exception as e:
        print(ticker, e)
        continue

final_df = pd.DataFrame(rows)

percentile_columns = [
    "One-Year Return Percentile",
    "Six-Month Return Percentile",
    "Three-Month Return Percentile",
    "One-Month Return Percentile"
]

final_df[percentile_columns] = final_df[percentile_columns].astype(float)

for period in [
    "One-Year Price Return",
    "Six-Month Price Return",
    "Three-Month Price Return",
    "One-Month Price Return"
]:
    percentile_col = period.replace("Price Return", "Return Percentile")

    final_df[percentile_col] = final_df[period].apply(
        lambda x: percentileofscore(
            final_df[period].dropna(),
            x
        ) / 100 if pd.notna(x) else np.nan
    )

position_size = portfolio_size / len(final_df.index)

final_df["Number of Shares To Buy"] = np.floor(
    position_size / final_df["Price"]
).astype("Int64")

final_df.sort_values(
    "One-Year Return Percentile",
    ascending=False,
    inplace=True
)

final_df


ABC single positional indexer is out-of-bounds
ABMD single positional indexer is out-of-bounds
ALXN single positional indexer is out-of-bounds
ANSS single positional indexer is out-of-bounds
ANTM single positional indexer is out-of-bounds
ATVI single positional indexer is out-of-bounds
BF.B single positional indexer is out-of-bounds
BLL single positional indexer is out-of-bounds
BRK.B single positional indexer is out-of-bounds
CERN single positional indexer is out-of-bounds
CMA single positional indexer is out-of-bounds
COG single positional indexer is out-of-bounds
CTL single positional indexer is out-of-bounds
CTXS single positional indexer is out-of-bounds
CXO single positional indexer is out-of-bounds
DFS single positional indexer is out-of-bounds
DISCA single positional indexer is out-of-bounds
DISCK single positional indexer is out-of-bounds
DISH single positional indexer is out-of-bounds
DRE single positional indexer is out-of-bounds
ETFC single positional indexer is out-of-boun

,Ticker,Price,Number of Shares To Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile
428,WDC,441.989990,5,8.932823,1.000000,1.908116,0.997783,0.641370,0.993348,0.498424,0.986696
376,STX,731.934998,3,6.965709,0.997778,1.936079,1.000000,0.751455,0.997783,0.704712,0.995565
289,MU,578.340027,3,6.181917,0.995556,1.654757,0.995565,0.524995,0.986696,0.579129,0.993348
213,INTC,96.669998,22,3.688167,0.993333,1.610586,0.993348,0.989095,1.000000,0.918817,0.997783
177,GLW,160.009995,13,2.549429,0.991111,0.887754,0.984479,0.461470,0.982262,0.081733,0.778271
...,...,...,...,...,...,...,...,...,...,...,...
83,CHTR,171.705002,12,-0.554013,0.008889,-0.222738,0.082040,-0.234075,0.055432,-0.218777,0.004435
165,FMC,14.650000,151,-0.595766,0.006667,0.104010,0.574279,-0.133605,0.206208,-0.174648,0.008869
162,FISV,63.029999,35,-0.658133,0.004444,-0.022033,0.314856,0.055425,0.694013,0.122329,0.869180
221,IT,145.384995,15,-0.659910,0.002222,-0.360017,0.013304,-0.078851,0.339246,-0.078967,0.090909


In [16]:
final_df

,Ticker,Price,Number of Shares To Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile
428,WDC,441.989990,5,8.932823,1.000000,1.908116,0.997783,0.641370,0.993348,0.498424,0.986696
376,STX,731.934998,3,6.965709,0.997778,1.936079,1.000000,0.751455,0.997783,0.704712,0.995565
289,MU,578.340027,3,6.181917,0.995556,1.654757,0.995565,0.524995,0.986696,0.579129,0.993348
213,INTC,96.669998,22,3.688167,0.993333,1.610586,0.993348,0.989095,1.000000,0.918817,0.997783
177,GLW,160.009995,13,2.549429,0.991111,0.887754,0.984479,0.461470,0.982262,0.081733,0.778271
...,...,...,...,...,...,...,...,...,...,...,...
83,CHTR,171.705002,12,-0.554013,0.008889,-0.222738,0.082040,-0.234075,0.055432,-0.218777,0.004435
165,FMC,14.650000,151,-0.595766,0.006667,0.104010,0.574279,-0.133605,0.206208,-0.174648,0.008869
162,FISV,63.029999,35,-0.658133,0.004444,-0.022033,0.314856,0.055425,0.694013,0.122329,0.869180
221,IT,145.384995,15,-0.659910,0.002222,-0.360017,0.013304,-0.078851,0.339246,-0.078967,0.090909


## Calculating the HQM Score

We'll now calculate our `HQM Score`, which is the high-quality momentum score that we'll use to filter for stocks in this investing strategy.

The `HQM Score` will be the arithmetic mean of the 4 momentum percentile scores that we calculated in the last section.

To calculate arithmetic mean, we will use the `mean` function from Python's built-in `statistics` module.

In [18]:
from statistics import mean

for row in final_df.index:
    momentum_percentiles = [
        final_df.loc[row, "One-Year Return Percentile"],
        final_df.loc[row, "Six-Month Return Percentile"],
        final_df.loc[row, "Three-Month Return Percentile"],
        final_df.loc[row, "One-Month Return Percentile"]
    ]
    
    final_df.loc[row, "HQM Score"] = mean(momentum_percentiles)

final_df.sort_values(
    "HQM Score",
    ascending=False,
    inplace=True
)

hqm_dataframe = final_df[:50]
hqm_dataframe.reset_index(drop=True, inplace=True)
position_size = portfolio_size / len(hqm_dataframe.index)

hqm_dataframe["Number of Shares To Buy"] = np.floor(
    position_size / hqm_dataframe["Price"]
).astype("Int64")

hqm_dataframe


C:\Users\nadir_i69tsxn\AppData\Local\Temp\ipykernel_1600\4251704793.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hqm_dataframe["Number of Shares To Buy"] = np.floor(


,Ticker,Price,Number of Shares To Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,STX,731.934998,27,6.965709,0.997778,1.936079,1.000000,0.751455,0.997783,0.704712,0.995565,0.997781
1,INTC,96.669998,206,3.688167,0.993333,1.610586,0.993348,0.989095,1.000000,0.918817,0.997783,0.996116
2,WDC,441.989990,45,8.932823,1.000000,1.908116,0.997783,0.641370,0.993348,0.498424,0.986696,0.994457
3,MU,578.340027,34,6.181917,0.995556,1.654757,0.995565,0.524995,0.986696,0.579129,0.993348,0.992791
4,PWR,748.309998,26,1.332412,0.966667,0.706680,0.968958,0.611075,0.991131,0.334766,0.975610,0.975591
5,KEYS,351.309998,56,1.365405,0.971111,0.961530,0.988914,0.585334,0.988914,0.206256,0.933481,0.970605
6,AMD,340.660004,58,2.447976,0.986667,0.362368,0.886918,0.701683,0.995565,0.566253,0.991131,0.965070
7,CAT,874.849976,22,1.732076,0.982222,0.604453,0.960089,0.266970,0.933481,0.222101,0.937916,0.953427
8,MCHP,94.830002,210,1.038528,0.953333,0.617626,0.962306,0.219332,0.904656,0.445579,0.984479,0.951194
9,TXN,280.885010,71,0.746420,0.913333,0.774086,0.973392,0.260026,0.931264,0.441397,0.982262,0.950063


## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

In [34]:
# initialize writer with engine
writer = pd.ExcelWriter('recommended_trades_momentum.xlsx', engine='xlsxwriter')
# Specify tabs
hqm_dataframe.to_excel(writer, 'Recommended Trades Momentum', index = False)

C:\Users\nadir_i69tsxn\AppData\Local\Temp\ipykernel_1600\557929491.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  hqm_dataframe.to_excel(writer, 'Recommended Trades Momentum', index = False)


## Creating the Formats We'll Need For Our .xlsx File

You'll recall from our first project that formats include colors, fonts, and also symbols like % and $. We'll need four main formats for our Excel document:

* String format for tickers
* \$XX.XX format for stock prices
* \$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

Since we already built our formats in the last section of this course, I've included them below for you. Run this code cell before proceeding.

In [35]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_template = writer.book.add_format(
        {
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

dollar_template = writer.book.add_format(
        {
            'num_format':'$0.00',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

integer_template = writer.book.add_format(
        {
            'num_format':'0',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

percent_template = writer.book.add_format(
        {
            'num_format':'0.0%',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

In [36]:
column_formats = {
'A': ['Ticker', string_template],
'B': ['Price', dollar_template],
'C': ['Number of Shares To Buy', integer_template],
'D': ['One-Year Price Return', percent_template],
'E': ['One-Year Price Return Percentile', percent_template],
'F': ['Six-Month Price Return', percent_template],
'G': ['Six-Month Price Return Percentile', percent_template],
'H': ['Three-Month Price Return Percentile', percent_template],
'I': ['Three-Month Price Return Percentile', percent_template],
'J': ['One-Month Price Return Percentile', percent_template],
'K': ['One-Month Price Return Percentile', percent_template],
'L': ['HQM Score', percent_template],
}

for column in column_formats.keys():
    writer.sheets['Recommended Trades Momentum'].set_column(f'{column}:{column}', 22, column_formats[column][1])
    writer.sheets['Recommended Trades Momentum'].write(f'{column}1', column_formats[column][0], column_formats[column][1])

writer.close()

## Saving Our Excel Output

As before, saving our Excel output is very easy: